# Augment service_tickets_21m → service_tickets_63m

Triplicate every row (3 copies total) and stamp each output row with a new
global sequential `id`. The FD attributes are copied verbatim, so every FD
pattern in `fds.csv` is preserved exactly — only the row count grows (~21M →
~63M). `id` is not part of any FD, so it does not affect the patterns. Used to
stress-test the pipeline's scalability (paper §6.4) on a larger table.

Streaming: we read the source one batch at a time with `iter_batches` and
write each tripled batch straight to disk, so peak memory is bounded by the
batch size — the full table is never materialised.

**Inputs → Outputs:** `service_tickets_21m/{clean.parquet, fds.csv}` →
`service_tickets_63m/{clean.parquet, fds.csv}`.

**Config knobs (next cell):**
- `src` / `dst` — source and augmented dataset parquet paths.
- `FACTOR` — copies per row (3 → ~63M rows).
- `BATCH` — rows read per streamed batch; bounds peak memory.

## Config

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

src = Path("..") / "datasets" / "service_tickets_21m" / "clean.parquet"
dst = Path("..") / "datasets" / "service_tickets_63m" / "clean.parquet"
dst.parent.mkdir(parents=True, exist_ok=True)

FACTOR = 3          # copies per source row (3 → ~63M rows)
BATCH = 250_000     # rows per streamed batch; bounds peak memory

## Augment (triplicate rows + reindex) and copy FDs

In [ ]:
# stream src → dst, one batch at a time. each batch is tripled by repeating
# every row FACTOR times (np.repeat keeps the 3 copies of a row adjacent),
# then prepended with a running global id. BATCH bounds peak memory: at most
# BATCH * FACTOR rows are held at once.
pf = pq.ParquetFile(src)
n_total = pf.metadata.num_rows

writer = None
written = 0
for batch in pf.iter_batches(batch_size=BATCH):
    n = batch.num_rows
    rep = np.repeat(np.arange(n), FACTOR)
    tbl = pa.Table.from_batches([batch]).take(pa.array(rep))
    ids = pa.array(np.arange(written, written + n * FACTOR, dtype=np.int64))
    tbl = tbl.add_column(0, "id", ids)
    if writer is None:
        writer = pq.ParquetWriter(dst, tbl.schema, compression="zstd")
    writer.write_table(tbl)
    written += n * FACTOR
    print(f"{written:,} / {n_total * FACTOR:,} rows written", end="\r")
writer.close()

pq_mb = dst.stat().st_size / 1024 / 1024
print(f"\n{dst.name}: {pq_mb:,.1f} MB, {written:,} rows")

In [ ]:
# fds.csv is unchanged — `id` is not part of any FD, so the new table reuses
# the source FDs verbatim.
shutil.copy(src.parent / "fds.csv", dst.parent / "fds.csv")

## Sanity check

In [ ]:
# sanity check: row count, schema, and that ids are unique & contiguous
import polars as pl

lf = pl.scan_parquet(dst)
n_rows = lf.select(pl.len()).collect().item()
print(f"{n_rows:,} rows, {len(lf.collect_schema().names())} cols")
lf.head(6).collect()